<a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/landingcollab/weightslab/examples/Notebooks/PyTorch/ws-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab image-classification notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> is an open-source PyTorch tool for dataset debugging, mislabel detection, and mid-training data curation. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details, and raise an issue on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a> for support.</div>

# Image Classification with WeightsLab

This notebook trains a small CNN on **MNIST** and instruments it with WeightsLab so every training signal is traced **back to the exact samples** producing it.

Most data problems (mislabels, outliers, class imbalance) stay invisible until your model tells you through the loss. By wrapping your training objects with `wl.watch_or_edit(...)`, WeightsLab records **per-sample** loss and metrics live, so you can rank the worst samples, spot bad labels, and curate the dataset **without restarting training**.

### What you'll do
1. Install WeightsLab.
2. Set every knob in one **config** dict (like a `config.yaml`).
3. Wrap the model, optimizer, dataloaders, loss, and metric with the SDK.
4. Train while per-sample signals are captured.
5. Rank the highest-loss samples inline, and (optionally) open the live **Weights Studio** UI.

## Setup

Install WeightsLab from PyPI. On Colab the free **T4 GPU** runtime is plenty for this demo (`Runtime -> Change runtime type -> T4 GPU`).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [4]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


In [5]:
%pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev3"

Looking in indexes: https://test.pypi.org/simple/, https://pypi.org/simple/
  Using cached torchmetrics-1.9.0-py3-none-any.whl.metadata (23 kB)
  Using cached onnx-1.20.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.4 kB)
  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_openai-1.3.5-py3-none-any.whl.metadata (3.4 kB)
  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.4.9-py3-none-any.whl.metadata (4.7 kB)
  Using cached openai-2.45.0-py3-none-any.whl.metadata (34 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase.

In [6]:
import os
import tempfile
import logging

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.examples.utils.baseline_models.pytorch.models import FashionCNN as CNN
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

15/07/2026-09:40:48.992 INFO:root:setup_logging: WeightsLab logging initialized - Log file: /tmp/tmpfqqv3mft/weightslab_logs/weightslab_20260715_094048.log
15/07/2026-09:40:48.994 INFO:weightslab:<module>: WeightsLab package initialized - Log level: INFO, Log to file: True
15/07/2026-09:40:48.995 INFO:weightslab:<module>: 
╭─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                     │
│  $$\      $$\           $$\           $$\         $$\               $$\                 $$\         │
│  $$ | $\  $$ |          \__|          $$ |        $$ |              $$ |                $$ |        │
│  $$ |$$$\ $$ | $$$$$$\  $$\  $$$$$$\  $$$$$$$\  $$$$$$\    $$$$$$$\ $$ |       $$$$$$\  $$$$$$$\    │
│  $$ $$ $$\$$ |$$  __$$\ $$ |$$  __$$\ $$  __$$\ \_$$  _|  $$  _____|$$ |       \____$$\ $$  __$$\   │
│  $$$$  _$$$$ |$$$$$$$$ |$$ |$$ /  $$ |$$ |  $$ | 

## 2. A dataset that carries sample identity

WeightsLab attributes every signal to a **sample id**. This thin wrapper around torchvision's MNIST returns `(image, id, label)` and attaches a virtual filepath per sample, so signals can be traced back to individual images in the UI.

In [30]:
class MNISTCustomDataset(Dataset):
    """MNIST that returns (image, index, label) and tracks a virtual filepath."""

    def __init__(self, root, train=True, download=False, transform=None):
        self.mnist = datasets.MNIST(root=root, train=train, download=download, transform=None)
        self.transform = transform
        self.train = train
        split = "train" if train else "test"
        self.filepaths = {}
        for idx in range(len(self.mnist)):
            label = self.mnist.targets[idx].item()
            self.filepaths[idx] = os.path.join(
                "MNIST", "processed", split, f"class_{label}", f"sample_{idx:05d}.pt"
            )

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        if self.transform:
            image = self.transform(image)
        return image, idx, label

## 3. Configuration

Every tunable lives here, in one dict, like a `config.yaml` with comments. Wrapping it with `flag="hyperparameters"` lets the Studio UI read (and live-edit) these values while training.

In [ ]:
log_dir = tempfile.mkdtemp(prefix="weightslab_mnist_")

config = {
    # -- Experiment -------------------------------------------------------
    "experiment_name": "mnist_classification",  # name shown in Weights Studio
    "device": str(device),                       # "cuda" if a GPU is available, else "cpu"
    "root_log_dir": log_dir,                      # where signal history / dataframes are written
    "num_classes": 10,                            # MNIST digits 0-9

    # -- Training schedule ------------------------------------------------
    "training_steps_to_do": 2000,                 # total optimizer steps (raise for a longer live run)
    "eval_full_to_train_steps_ratio": 100,        # run a full eval every N steps
    "write_export_ratio": 100,                    # export signal history + dataframe every N steps

    # -- Optimizer --------------------------------------------------------
    "learning_rate": 0.01,                        # Adam learning rate

    # -- Data loaders -----------------------------------------------------
    # One block per loader, keyed by loader_name. EVERY kwarg passed to
    # wl.watch_or_edit(..., flag="data") lives here so the wrap cell below stays
    # declarative — no dataloader settings hardcoded in the cells. Add a new
    # loader by adding another block here.
    "data": {
        "train_loader": {
            "batch_size": 16,                     # training batch size
            "shuffle": True,                      # shuffle each epoch
            "is_training": True,                  # marks this loader as the training split
            "compute_hash": False,                # skip per-sample content hashing (faster init)
            "preload_labels": True,               # preload labels into the ledger at startup
            "preload_metadata": False,            # load metadata lazily on first access
            "enable_h5_persistence": True,        # persist per-sample stats to the H5 store
        },
        "test_loader": {
            "batch_size": 128,                    # evaluation batch size
            "shuffle": False,                     # keep test order stable
            "is_training": False,                 # marks this loader as an eval split
            "compute_hash": False,
            "preload_labels": True,
            "preload_metadata": False,
            "enable_h5_persistence": True,
        },
    },

    # -- Services ---------------------------------------------------------
    "serving_grpc": True,                         # expose the gRPC backend for Weights Studio

    # -- Live UI tunnel (see the Expose section) -------------------------
    "expose_ui": True,                            # open a bore tunnel so a local Studio can connect
    "backend_port": 50051,                        # gRPC port the tunnel forwards
}

wl.watch_or_edit(config, flag="hyperparameters", poll_interval=1.0)
print("Experiment logs ->", log_dir)

## 4. Wrap the training objects

This is the heart of WeightsLab. Each object is passed through `wl.watch_or_edit(...)` with a `flag` describing its role. The returned objects behave exactly like the originals, but now report their state and per-sample signals to WeightsLab.

In [ ]:
data_root = os.path.join(log_dir, "data")
os.makedirs(data_root, exist_ok=True)

train_ds = MNISTCustomDataset(root=data_root, train=True, download=True,
                              transform=transforms.ToTensor())
test_ds = MNISTCustomDataset(root=data_root, train=False, download=True,
                             transform=transforms.ToTensor())

# Model + optimizer
model = wl.watch_or_edit(CNN().to(device), flag="model", device=device)
optimizer = wl.watch_or_edit(
    optim.Adam(model.parameters(), lr=config["learning_rate"]), flag="optimizer")

# Tracked dataloaders — all loader settings come from config["data"][<loader_name>],
# so nothing is hardcoded here. Unpack each block straight into the wrapper.
train_loader = wl.watch_or_edit(
    train_ds, flag="data", loader_name="train_loader",
    **config["data"]["train_loader"],
)
test_loader = wl.watch_or_edit(
    test_ds, flag="data", loader_name="test_loader",
    **config["data"]["test_loader"],
)

# Watched losses + metric (they log themselves per sample)
train_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                   flag="loss", signal_name="train-loss-CE", log=True)
test_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                  flag="loss", signal_name="test-loss-CE", log=True)
metric = wl.watch_or_edit(
    Accuracy(task="multiclass", num_classes=config["num_classes"]).to(device),
    flag="metric", signal_name="metric-ACC", log=True)

## 5. Train and evaluate steps

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab which phase it's in. `criterion(..., batch_ids=ids, preds=preds)` passes the sample ids so the loss is stored **per sample**, and `wl.save_signals(...)` logs a custom per-sample accuracy signal during evaluation.

In [47]:
def train(loader, model, optimizer, criterion, device):
    with guard_training_context:
        inputs, ids, labels = next(loader)
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        preds = logits.argmax(dim=1, keepdim=True)

        loss = criterion(logits.float(), labels.long(), batch_ids=ids, preds=preds)
        total_loss = loss.mean()
        total_loss.backward()
        optimizer.step()
    return total_loss.detach().cpu().item()


import time
def test(loader, model, criterion, metric, device, n_batches):
    losses = torch.tensor(0.0, device=device)
    st = time.time()
    for inputs, ids, labels in loader:
        with guard_testing_context:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1, keepdim=True)

            loss = criterion(logits, labels, batch_ids=ids, preds=preds)
            losses += loss.mean()
            metric.update(logits, labels)

            correct = (preds.view(-1) == labels.view(-1)).float()
            wl.save_signals(
                preds_raw=logits, targets=labels, batch_ids=ids, preds=preds,
                signals={
                    "test_metric/Accuracy_per_sample": correct,
                    "test_metric/Inverse_Accuracy_per_sample": 1.0 - correct,
                },
            )
    print(f"Compute test in {time.time()-st}s")
    return (losses / n_batches).item(), (metric.compute() * 100).item()

## 6. (Optional) Expose the backend for the live UI

Skip this if you only want the inline results below. To watch training **live in Weights Studio on your own machine**, keep `config["expose_ui"] = True`.

This opens a **raw-TCP** tunnel over [`bore`](https://github.com/ekzhang/bore) and its free public relay (`bore.pub`) - **no signup, no account, no credit card** - and prints the exact `weightslab tunnel ...` command to run locally.

> Note: raw TCP is required so gRPC's HTTP/2 frames pass through untouched (this is also why we avoid `ngrok`, which now needs a credit card for TCP endpoints).
>
> Note: `bore.pub` is a shared public relay; the random port is the only thing protecting your endpoint. Fine for a demo, not for sensitive data.

In [48]:
import re, tarfile, threading, urllib.request, subprocess

endpoint = None
if config["expose_ui"]:
    BORE = "v0.6.0"
    if not os.path.exists("bore"):
        urllib.request.urlretrieve(
            f"https://github.com/ekzhang/bore/releases/download/{BORE}/bore-{BORE}-x86_64-unknown-linux-musl.tar.gz",
            "bore.tar.gz")
        with tarfile.open("bore.tar.gz") as t:
            t.extractall()
        os.chmod("bore", 0o755)

    proc = subprocess.Popen(
        ["./bore", "local", str(config["backend_port"]), "--to", "bore.pub"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        m = re.search(r"bore\.pub:(\d+)", line)
        if m:
            endpoint = f"bore.pub:{m.group(1)}"
            break
    threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()

    print("=" * 60)
    print("Backend exposed at:", endpoint)
    print("On your OWN machine (Docker running), run these two commands:")
    print("    weightslab ui launch")
    print(f"    weightslab tunnel {endpoint}")
    print("=" * 60)
else:
    print("expose_ui is False - inline results only, no tunnel.")

Backend exposed at: bore.pub:25880
On your OWN machine (Docker running), run these two commands:
    weightslab ui launch
    weightslab tunnel bore.pub:25880


## 7. Serve and train

`wl.serve(serving_grpc=True)` starts the background gRPC server (non-blocking) that Weights Studio connects to. `wl.start_training(...)` flips the experiment into the *training* state, then we run the loop, periodically evaluating and exporting signals.

Leave this cell **running** while you watch it stream in the UI.

In [49]:
wl.serve(serving_grpc=config["serving_grpc"])

15/07/2026-10:13:47.891 INFO:weightslab.trainer.trainer_services:_run_security_preflight: 

# #######################################
# #######################################
[gRPC] Security preflight checks:
	TLS: DISABLED (unencrypted traffic)
	Auth tokens: NONE configured
	! WARNING: GRPC_TLS_ENABLED=0. Traffic will be unencrypted. Use only for development.
	! WARNING: No GRPC_AUTH_TOKEN/GRPC_AUTH_TOKENS configured. Only transport-level trust (TLS/mTLS) will protect RPC access.
# #######################################
# #######################################

15/07/2026-10:13:47.894 INFO:weightslab.trainer.trainer_services:grpc_serve: [gRPC] Watchdogs disabled via WEIGHTSLAB_DISABLE_WATCHDOGS.
15/07/2026-10:13:47.895 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Thread callback started
15/07/2026-10:13:47.896 INFO:weightslab.trainer.trainer_services:grpc_serve: [gRPC] Server started with watchdogs disabled (host=0.0.0.0 port=50051 workers=None)
15/07/20

In [50]:
wl.start_training(timeout=3)

steps = config["training_steps_to_do"]
eval_ratio = config["eval_full_to_train_steps_ratio"]
export_ratio = config["write_export_ratio"]
n_test_batches = len(test_loader)

test_loss = test_acc = None
pbar = tqdm(range(steps), desc="Training")
for step in pbar:
    age = model.get_age() if hasattr(model, "get_age") else step

    train_loss = train(train_loader, model, optimizer, train_criterion, device)

    if age > 0 and age % eval_ratio == 0:
        test_loss, test_acc = test(test_loader, model, test_criterion, metric, device, n_test_batches)

    if age > 0 and age % export_ratio == 0:
        wl.write_history()
        wl.write_dataframe()

    postfix = {"loss": f"{train_loss:.3f}"}
    if test_acc is not None:
        postfix["test_acc"] = f"{test_acc:.1f}%"
    pbar.set_postfix(postfix)

wl.write_history()
wl.write_dataframe()
print("Training complete. Logs at:", log_dir)

15/07/2026-10:13:48.997 INFO:weightslab.src:start_training: Starting WeightsLab training mode with a timeout of 3 seconds.


[PreviewCache]: 100%|██████████| 2000/2000 [00:01<00:00, 1151.83sample/s]

15/07/2026-10:13:50.011 INFO:weightslab.trainer.services.data_service:_build_preview_cache: [PreviewCache] Done: 2000/2000 cached in 1.7s (0.9 ms/sample)


15/07/2026-10:13:52.001 INFO:weightslab.components.global_monitoring:resume: 
Attempting to resume training...
15/07/2026-10:13:52.110 INFO:weightslab.components.global_monitoring:resume: Hashes by module: ['1d3f2679', '1e11f32f', '9e70c5a5']
15/07/2026-10:13:52.113 INFO:weightslab.components.global_monitoring:resume: Resuming training now...
15/07/2026-10:13:52.114 INFO:weightslab.components.global_monitoring:resume: Hashes by module on resume: ['1d3f2679', '1e11f32f', '9e70c5a5']
15/07/2026-10:13:52.115 INFO:weightslab.components.global_monitoring:resume: 
Training resumed as modules hashes have been computed: ['1d3f2679', '1e11f32f', '9e70c5a5'].


Training:   0%|          | 0/2000 [00:00<?, ?it/s]

15/07/2026-10:13:56.840 INFO:weightslab.trainer.services.data_service:GetDataSplits: GetDataSplits returning: ['test_loader', 'train_loader']
15/07/2026-10:13:57.827 INFO:weightslab.components.checkpoint_manager:save_model_checkpoint: Saved model checkpoint: 1d3f26791e11f32f9e70c5a5_step_000800.pt
15/07/2026-10:13:58.976 WARNING:weightslab.trainer.services.data_service:_slowUpdateInternals: [LockWatchdog] _update_lock[_slowUpdateInternals]   RELEASED by WL-ViewRefresh                 held 2134.7 ms ← SLOW
15/07/2026-10:13:59.209 WARNING:weightslab.trainer.services.data_service:_slowUpdateInternals: [LockWatchdog] _update_lock[_slowUpdateInternals]   RELEASED by WL-ViewRefresh                 held 1760.2 ms ← SLOW
15/07/2026-10:13:59.731 WARNING:weightslab.trainer.services.experiment_service:_get_latest_logger_data_impl: get_signal_history() took 1341.5ms (slow — possible lock contention)
15/07/2026-10:14:00.237 INFO:weightslab.components.checkpoint_manager:save_logger_snapshot: Saved l

KeyboardInterrupt: 

## 8. Which samples hurt the most?

WeightsLab exports a per-sample dataframe to `root_log_dir`. Ranking by loss surfaces the samples the model struggles with - the usual suspects for **mislabels and outliers**. This is the same signal the Studio UI visualizes; here we peek at it inline.

In [ ]:
import glob
import pandas as pd

paths = sorted(glob.glob(os.path.join(log_dir, "**", "*.json"), recursive=True),
               key=os.path.getmtime)
if paths:
    df = pd.read_json(paths[-1])
    print(f"Loaded {paths[-1]}  (shape={df.shape})")
    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    if loss_cols:
        display(df.sort_values(loss_cols[-1], ascending=False).head(50))
    else:
        display(df.head())
else:
    print("No export found - run the training cell first.")

## 9. See it live in Weights Studio

Everything above ran headless. The real payoff is the **Weights Studio** UI, where you browse the highest-loss images, filter by class, and curate the dataset mid-training.

Studio runs as a local Docker stack (a static frontend + an Envoy gRPC-Web proxy). **Colab has no Docker daemon**, so you don't run the UI *inside* Colab - you run Studio on your own machine and point it at this notebook's backend using the `bore.pub:<port>` endpoint **printed in Section 6**.

**On your machine** (with Docker Desktop):
```bash
pip install weightslab

# Terminal 1 - launch the UI (plaintext HTTP, the default)
weightslab ui launch                       # opens http://localhost:5173

# Terminal 2 - bridge the Colab backend to localhost:50051
weightslab tunnel bore.pub:12345           # the host:port printed in Section 6
```

Then open **http://localhost:5173** - Studio connects through your local Envoy -> `weightslab tunnel` -> `bore.pub` -> this Colab backend, and you watch training stream live.

> Note: keep it plaintext end-to-end (the default `weightslab ui launch`, raw-TCP `bore`) so gRPC's HTTP/2 frames pass through untouched.

Prefer to keep it all local? Run this same example on your own machine (`weightslab start example --cls`) and launch the UI next to it - no tunnel required.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>